# Glitch × ShortGPT — Pexels-free Google Gemini video studio

This notebook runs ShortGPT in a **Google Colab cloud runtime**. Nothing is installed on your PC.

## Free setup
- One Google AI Studio Gemini API key: https://aistudio.google.com/app/apikey
- Choose **Edge TTS** inside ShortGPT for free voice generation. Do not select ElevenLabs unless you independently have a key.

**No Pexels key is requested or needed.** This notebook patches ShortGPT's video pipeline to render a locally-created cinematic gradient background with captions and narration instead of fetching stock footage. Gemini remains responsible for script generation and metadata; it does not provide stock-video footage through this integration.

Colab runtimes are temporary and can disconnect when idle or reclaimed. Restart this notebook when that happens; the public Gradio URL may change.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg imagemagick
!sed -i '/<policy domain="path" rights="none" pattern="@\*"/d' /etc/ImageMagick-6/policy.xml 2>/dev/null || true
!ffmpeg -version | head -n 1


In [ ]:
import os, pathlib, subprocess, sys, textwrap
REPO = "/content/ShortGPT"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "stable", "https://github.com/RayVentura/ShortGPT.git", REPO], check=True)
os.chdir(REPO)
packages = ["gradio==5.12.0", "gradio_client==1.5.4", "openai==1.37.0", "httpx==0.27.2", "tiktoken", "tinydb", "tinymongo", "proglog", "yt-dlp>=2025.1.12", "whisper-timestamped", "protobuf==3.20.3", "pillow==10.4.0", "moviepy==2.1.2", "progress", "questionary", "edge-tts"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

# Keep the upstream Gemini integration, but update its retired model identifier.
gpt_utils_path = pathlib.Path("shortGPT/gpt/gpt_utils.py")
gpt_utils_source = gpt_utils_path.read_text(encoding="utf-8")
gpt_utils_source = gpt_utils_source.replace("gemini-2.0-flash-lite-preview-02-05", "gemini-3.5-flash-lite")
gpt_utils_path.write_text(gpt_utils_source, encoding="utf-8")

# Remove ShortGPT's hard dependency on Pexels. The stock-footage search steps
# become no-ops; the renderer receives a locally generated cinematic backdrop.
engine_path = pathlib.Path("shortGPT/engine/content_video_engine.py")
engine_source = engine_path.read_text(encoding="utf-8")
engine_source = engine_source.replace("from shortGPT.api_utils.pexels_api import getBestVideo\\n", "")
start = engine_source.index("    def _generateVideoSearchTerms(self):")
end = engine_source.index("    def _chooseBackgroundMusic(self):", start)
engine_source = engine_source[:start] + """    def _generateVideoSearchTerms(self):
        # Stock-footage search is intentionally disabled in the Pexels-free build.
        self._db_timed_video_searches = []

    def _generateVideoUrls(self):
        # No external stock-footage provider or API key is required.
        self._db_timed_video_urls = []

"""+engine_source[end:]
engine_source = engine_source.replace("import shutil\\n", "import shutil\\nimport subprocess\\n")
needle = """            videoEditor.addEditingStep(EditingStep.ADD_VOICEOVER_AUDIO, {
                                       'url': self._db_audio_path})
"""
replacement = needle + """            # Build a cinematic, locally rendered background (no stock API).
            import math
            from PIL import Image, ImageDraw, ImageFilter
            width, height = (1080, 1920) if self._db_format_vertical else (1920, 1080)
            bg_path = os.path.join(self.dynamicAssetDir, "glitch_gradient_background.png")
            bg_video_path = os.path.join(self.dynamicAssetDir, "glitch_gradient_background.mp4")
            os.makedirs(self.dynamicAssetDir, exist_ok=True)
            base = Image.new("RGB", (width, height))
            pixels = base.load()
            for y in range(height):
                t = y / max(1, height - 1)
                for x in range(width):
                    glow = max(0.0, 1.0 - math.sqrt(((x / width - 0.72) * 1.2) ** 2 + ((t - 0.22) * 1.4) ** 2))
                    pixels[x, y] = (int(8 + 10 * (1-t) + 12*glow), int(13 + 30*glow), int(24 + 38*glow))
            overlay = Image.new("RGBA", (width, height), (0, 0, 0, 0))
            draw = ImageDraw.Draw(overlay)
            draw.ellipse((int(width*.52), int(height*.02), int(width*1.12), int(height*.62)), fill=(34, 211, 160, 58))
            draw.ellipse((-int(width*.28), int(height*.48), int(width*.48), int(height*1.12)), fill=(59, 130, 246, 38))
            overlay = overlay.filter(ImageFilter.GaussianBlur(max(35, width//18)))
            base = Image.alpha_composite(base.convert("RGBA"), overlay).convert("RGB")
            base.save(bg_path, quality=92)
            duration = max(1.0, float(self._db_voiceover_duration or 1.0))
            subprocess.run(["ffmpeg", "-y", "-loop", "1", "-i", bg_path, "-t", str(duration),
                            "-vf", f"scale={width}:{height},format=yuv420p", "-r", "25",
                            "-c:v", "libx264", "-preset", "ultrafast", bg_video_path],
                           check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
            videoEditor.addEditingStep(EditingStep.ADD_BACKGROUND_VIDEO, {
                "url": bg_video_path, "set_time_start": 0, "set_time_end": duration
            })
"""
if needle not in engine_source:
    raise RuntimeError("Could not find the expected ShortGPT voiceover render block; stopping safely.")
engine_source = engine_source.replace(needle, replacement, 1)
engine_path.write_text(engine_source, encoding="utf-8")

# The UI must not reject a valid Gemini-only setup.
ui_path = pathlib.Path("gui/ui_tab_video_automation.py")
ui_source = ui_path.read_text(encoding="utf-8")
old_check = """        pexels_api_key = ApiKeyManager.get_api_key("PEXELS_API_KEY")
        if not pexels_api_key:
            return "Your Pexels API key is missing. Please go to the config tab and enter the API key."
"""
if old_check not in ui_source:
    raise RuntimeError("Could not find the expected Pexels validation block; stopping safely.")
ui_path.write_text(ui_source.replace(old_check, "", 1), encoding="utf-8")

print("Installed ShortGPT dependencies.")
print("Patched video creation to use a locally rendered cinematic background; Pexels is no longer required.")
print("Gemini model configuration updated. Colab's preinstalled PyTorch/Torchaudio are reused.")


In [ ]:
from getpass import getpass
from shortGPT.config.api_db import ApiKeyManager, ApiProvider

gemini_key = getpass("Google AI Studio Gemini API key (input hidden): ").strip()
if not gemini_key:
    raise ValueError("A Gemini API key is required. Create one at https://aistudio.google.com/app/apikey")

ApiKeyManager.set_api_key(ApiProvider.GEMINI, gemini_key)
os.environ["GEMINI_API_KEY"] = gemini_key
del gemini_key
print("Gemini key stored in this temporary Colab runtime. It is not sent to Glitch.")
print("Pexels is disabled; ShortGPT will use its generated cinematic background, captions, and Edge TTS narration.")


In [ ]:
from gui.gui_gradio import ShortGptUI
print("Starting ShortGPT. Wait for the public https://....gradio.live link below.")
app = ShortGptUI(colab=True)
app.launch()
